# 5장. Pandas 기초

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 2장, 4장 |
| 이 장의 결과물 | 데이터셋을 DataFrame으로 불러와 결측치·빈 문서·중복을 정리하는 코드, 카테고리 분포 표 |
| 과제 연결 | 데이터 준비 체크리스트 "500개 이상, 2개 이상 카테고리, 결측치 제거 완료" |

**학습 목표**
- Series와 DataFrame의 구조를 알고 열을 고르고 조건으로 행을 거를 수 있다.
- TSV/CSV/JSON 파일과 sklearn 데이터셋을 DataFrame으로 불러올 수 있다.
- `value_counts`, `groupby`로 카테고리별 문서 수와 통계를 낼 수 있다.
- 카테고리마다 같은 수의 문서를 무작위로 뽑을 수 있다.
- 결측치, 빈 문자열, 중복 문서, 레이블이 엇갈린 중복을 찾아 지울 수 있다.

> NumPy가 "숫자 격자"라면 Pandas는 **이름 붙은 열을 가진 표**입니다. 텍스트와 레이블처럼 자료형이 다른 열을 함께 다룰 때 씁니다. TF-IDF 계산은 NumPy로, 데이터 준비와 결과 정리는 Pandas로 합니다.

In [1]:
# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

WORK = Path(tempfile.mkdtemp())   # 이 장에서 만드는 파일은 임시 폴더에 저장한다

## 5.1 Series와 DataFrame

- **Series**: 이름표(인덱스)가 붙은 1차원 배열. 열 하나입니다.
- **DataFrame**: 같은 인덱스를 공유하는 Series 여러 개를 묶은 표입니다.

In [2]:
mini = pd.DataFrame({"text": MINI_DOCS, "label": MINI_LABELS})
mini

,text,label
0,The rocket launched into space!,space
1,NASA launched a new space telescope.,space
2,The pitcher threw a fast ball.,baseball
3,Our team won the baseball game.,baseball
4,Astronauts watched the baseball game in space.,space


In [3]:
print("shape  :", mini.shape)            # (행 수, 열 수)
print("columns:", list(mini.columns))
print("index  :", list(mini.index))      # 행 이름표. 기본은 0부터 번호 = 문서 ID로 쓰기 좋다
print("dtypes :\n", mini.dtypes)

shape  : (5, 2)
columns: ['text', 'label']
index  : [0, 1, 2, 3, 4]
dtypes :
 text     str
label    str
dtype: object


### 열 하나 = Series, `.str`로 문자열 함수 한 번에 적용
텍스트 열에 `.str.메서드()`를 붙이면 모든 행에 문자열 메서드를 적용합니다. NumPy의 벡터화처럼 루프가 필요 없습니다.

In [4]:
text = mini["text"]                       # Series
print(type(text).__name__)
print(text.str.lower().head(3).tolist())
print("글자 수 :", text.str.len().tolist())
print("단어 수 :", text.str.split().str.len().tolist())
print("space 포함:", text.str.contains("space", case=False).tolist())

Series
['the rocket launched into space!', 'nasa launched a new space telescope.', 'the pitcher threw a fast ball.']
글자 수 : [31, 36, 30, 31, 46]
단어 수 : [5, 6, 6, 6, 7]
space 포함: [True, True, False, False, True]


## 5.2 파일과 데이터셋 불러오기

### TSV/CSV: `read_csv`
구분자는 `sep`로 지정합니다. 쉼표는 `","`(기본값), **탭은 `"\t"`**입니다.

> 과제 문서의 NSMC 예시 `sep='    '`(공백 4칸)는 탭 문자가 문서로 옮겨지며 공백으로 바뀐 것입니다. 실제로는 `sep="\t"`를 씁니다.

연습용으로 **지저분한** 한국어 뉴스 제목 데이터를 만들어 TSV로 저장한 뒤 다시 읽어 봅니다. 실제 데이터에서 자주 만나는 문제를 일부러 넣었습니다.

In [5]:
raw_rows = [
    ("손흥민, 시즌 10호골 폭발", "스포츠"),
    ("코스피 2,600선 회복…외국인 순매수", "경제"),
    ("정부, 반도체 지원 예산 확대", "경제"),
    ("누리호 4차 발사 성공", "IT과학"),
    ("손흥민, 시즌 10호골 폭발", "스포츠"),          # ① 완전히 같은 중복
    ("누리호  4차 발사   성공", "IT과학"),          # ② 공백만 다른 중복
    ("삼성전자 신형 스마트폰 공개", "IT과학"),
    ("삼성전자 신형 스마트폰 공개", "경제"),        # ③ 같은 제목, 다른 레이블
    (None, "정치"),                               # ④ 본문 결측
    ("", "사회"),                                 # ⑤ 빈 문자열
    ("   ", "사회"),                              # ⑥ 공백만 있는 문서
    ("국회, 예산안 처리 합의", "정치"),
    ("프로야구 개막전 매진 행렬", "스포츠"),
    ("폭염 특보 전국 확대", "사회"),
    ("AI 반도체 수출 사상 최대", None),            # ⑦ 레이블 결측
]
tsv_path = WORK / "messy_news.tsv"
pd.DataFrame(raw_rows, columns=["text", "label"]).to_csv(tsv_path, sep="\t", index=False)

messy = pd.read_csv(tsv_path, sep="\t", encoding="utf-8")
messy

,text,label
0,"손흥민, 시즌 10호골 폭발",스포츠
1,"코스피 2,600선 회복…외국인 순매수",경제
2,"정부, 반도체 지원 예산 확대",경제
3,누리호 4차 발사 성공,IT과학
4,"손흥민, 시즌 10호골 폭발",스포츠
5,누리호 4차 발사 성공,IT과학
6,삼성전자 신형 스마트폰 공개,IT과학
7,삼성전자 신형 스마트폰 공개,경제
8,NaN,정치
9,NaN,사회


- 파일에 쓸 때 `None`과 빈 문자열 `""`는 둘 다 빈칸이 되고, 다시 읽으면 **둘 다 `NaN`(결측)**이 됩니다(8번, 9번 행). 공백만 있는 문자열 `"   "`은 결측이 아니라 그대로 남습니다(10번 행). 이런 차이 때문에 결측과 빈 문서를 **따로** 검사해야 합니다.
- 한글 파일은 `encoding="utf-8"`을 명시합니다. 엑셀에서 열 CSV는 `"utf-8-sig"`로 저장하면 한글이 깨지지 않습니다.

### JSON: `read_json`
KLUE-YNAT 원본은 `[{...}, {...}]` 형태의 JSON 파일입니다. 같은 모양의 작은 파일로 연습합니다.

In [6]:
import json

json_path = WORK / "ynat_sample.json"
json_path.write_text(json.dumps([
    {"guid": "ynat-v1_train_00000", "title": "유튜브 내달 2일까지 크리에이터 지원 공간 운영", "label": "생활문화"},
    {"guid": "ynat-v1_train_00001", "title": "어버이날 맑다가 흐려져…남부지방 옅은 황사", "label": "생활문화"},
    {"guid": "ynat-v1_train_00002", "title": "내년부터 국가RD 평가 때 논문건수는 반영 않는다", "label": "사회"},
], ensure_ascii=False), encoding="utf-8")

ynat = pd.read_json(json_path)
ynat.rename(columns={"title": "text"})[["text", "label"]]

,text,label
0,유튜브 내달 2일까지 크리에이터 지원 공간 운영,생활문화
1,어버이날 맑다가 흐려져…남부지방 옅은 황사,생활문화
2,내년부터 국가RD 평가 때 논문건수는 반영 않는다,사회


### sklearn 내장 데이터셋: 20 Newsgroups
과제에서 쓰는 영어 데이터입니다. `fetch_20newsgroups`는 처음 한 번 인터넷에서 받아 홈 폴더의 `scikit_learn_data/`에 저장하고, 다음부터는 저장된 파일을 씁니다.

`remove=("headers", "footers", "quotes")`를 주는 이유는 7장에서 자세히 다룹니다. 간단히 말하면 헤더에 **작성자 이메일(개인정보)**과 **게시판 이름(=정답)**이 들어 있기 때문입니다.

In [7]:
from sklearn.datasets import fetch_20newsgroups

bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                           remove=("headers", "footers", "quotes"), random_state=42)
news = pd.DataFrame({
    "text": bunch.data,
    "label": [bunch.target_names[t] for t in bunch.target],
})
print(news.shape)
news.head()

(4941, 2)


,text,label
0,"According to the OSF/Motif Style Guide, one sh...",comp.windows.x
1,\nHow was this trade bad? I seem to recall Rat...,rec.sport.hockey
2,\nHe's not gone yet. The position opening is ...,rec.sport.hockey
3,Does anybody know what is going on with Lankfo...,rec.sport.baseball
4,"So what is your definition of ""interfering wit...",rec.sport.baseball


`bunch.target`은 카테고리 번호(0~4)이고 `bunch.target_names`가 번호 → 이름 표입니다. 사람이 읽기 쉽도록 이름으로 바꿔 열에 넣었습니다.

## 5.3 열 선택, 조건 필터, value_counts

| 문법 | 결과 |
|---|---|
| `df["text"]` | 열 하나 (Series) |
| `df[["text", "label"]]` | 열 여러 개 (DataFrame) |
| `df[조건]` | 조건이 True인 행만 |
| `df[(조건1) & (조건2)]` | 그리고. **괄호 필수**, `and` 대신 `&` |
| `df[(조건1) \| (조건2)]` | 또는. `or` 대신 `\|` |
| `df[df["label"].isin([...])]` | 목록 중 하나에 해당 |
| `df.loc[행 이름, 열 이름]` | 이름으로 고르기 |
| `df.iloc[행 번호, 열 번호]` | 위치로 고르기 |

In [8]:
space = news[news["label"] == "sci.space"]
print("sci.space 문서 수:", len(space))

sports = news[news["label"].isin(["rec.sport.baseball", "rec.sport.hockey"])]
print("스포츠 문서 수   :", len(sports))

long_space = news[(news["label"] == "sci.space") & (news["text"].str.len() > 3000)]
print("긴 sci.space 문서:", len(long_space))

print("\n첫 문서의 앞 200자:\n", news.iloc[0]["text"][:200])

sci.space 문서 수: 987
스포츠 문서 수   : 1993
긴 sci.space 문서: 71

첫 문서의 앞 200자:
 According to the OSF/Motif Style Guide, one should use cursor shapes to give
the user a visual clue of what is happening or what is expected of him. So
a "hourglass" cursor should be shown when the ap


### value_counts: 카테고리 분포
분류 문제에서 가장 먼저 확인할 것은 **카테고리별 문서 수가 고른지**입니다. 한쪽이 압도적으로 많으면 정확도만으로는 성능을 판단하기 어렵습니다(19장).

In [9]:
counts = news["label"].value_counts()
ratio = news["label"].value_counts(normalize=True)
pd.DataFrame({"문서 수": counts, "비율": ratio.round(3)})

,문서 수,비율
label,,
rec.sport.hockey,999,0.202
rec.sport.baseball,994,0.201
comp.windows.x,988,0.200
sci.space,987,0.200
comp.graphics,973,0.197


5개 카테고리가 각각 약 20%로 고르게 분포합니다.

## 5.4 groupby와 카테고리별 샘플링

`groupby("label")`은 레이블이 같은 행끼리 묶어서, 묶음마다 계산합니다.

In [10]:
news["n_chars"] = news["text"].str.len()
news["n_words"] = news["text"].str.split().str.len()

news.groupby("label")[["n_chars", "n_words"]].agg(["mean", "median", "max"]).round(0)

n_chars               n_words              
                      mean median    max    mean median    max
label                                                         
comp.graphics       1294.0  380.0  61278   197.0   65.0   9196
comp.windows.x      1475.0  479.0  65424   219.0   80.0   9365
rec.sport.baseball   724.0  392.0  15161   125.0   69.0   2479
rec.sport.hockey    1126.0  491.0  74878   185.0   86.0  11251
sci.space           1213.0  505.0  56793   190.0   85.0   9113

평균이 중앙값보다 훨씬 큽니다. 아주 긴 글 몇 개가 평균을 끌어올린다는 뜻입니다. 문서 길이 차이가 크다는 것은 3장에서 본 **L2 정규화가 필요한 이유**와 이어집니다.

### 카테고리마다 같은 수만큼 뽑기
데이터가 너무 크거나 카테고리마다 개수가 다를 때, 카테고리별로 같은 수를 무작위로 뽑습니다. 한국어 KLUE-YNAT은 학습 파일 4만 5천여 건에서 주제별로 1,000건씩 뽑아 7,000건을 씁니다. `random_state`를 고정해야 다시 실행해도 같은 문서가 뽑힙니다.

In [11]:
sampled = news.groupby("label").sample(n=100, random_state=42)
print(sampled["label"].value_counts())

again = news.groupby("label").sample(n=100, random_state=42)
print("\n다시 뽑아도 같은 문서?", sampled.index.equals(again.index))

label
comp.graphics         100
comp.windows.x        100
rec.sport.baseball    100
rec.sport.hockey      100
sci.space             100
Name: count, dtype: int64

다시 뽑아도 같은 문서? True


> 어떤 카테고리의 문서가 `n`보다 적으면 `sample`이 에러를 냅니다. `n=min(n, 가장 작은 카테고리 크기)`로 맞추거나 데이터를 먼저 확인하세요.

## 5.5 데이터 정리: 결측치, 빈 문서, 중복

5.2절의 지저분한 데이터를 단계별로 정리하면서, **각 단계에서 몇 건이 빠졌는지 기록**합니다. 이 기록은 리포트의 "데이터 준비" 부분에 그대로 쓸 수 있습니다.

### ① 결측치

In [12]:
print("열별 결측치 수:\n", messy.isna().sum(), sep="")

log = [("원본", len(messy))]
step1 = messy.dropna(subset=["text", "label"])
log.append(("본문 또는 레이블 결측 제거", len(step1)))

열별 결측치 수:
text     2
label    1
dtype: int64


### ② 빈 문서와 공백만 있는 문서
`dropna`로는 `"   "`처럼 공백만 있는 문서를 못 지웁니다. 앞뒤 공백을 지운 뒤 빈 문자열인지 확인합니다.

In [13]:
blank = step1["text"].str.strip() == ""
print("공백뿐인 문서:", step1.loc[blank, "text"].tolist())
step2 = step1[~blank]                     # ~ 는 True/False 뒤집기 (NOT)
log.append(("빈 문서 제거", len(step2)))

공백뿐인 문서: ['   ']


### ③ 중복과 레이블이 엇갈린 중복
`drop_duplicates()`는 **글자가 완전히 같은** 행만 중복으로 봅니다. `"누리호 4차 발사 성공"`과 `"누리호  4차 발사   성공"`은 공백만 다른데도 다른 문서로 취급합니다. 그래서 공백을 하나로 정리한 **비교용 열**을 만들어 그 열로 중복을 찾습니다.

같은 본문인데 레이블이 다르면(③번 행) 어느 쪽이 정답인지 알 수 없으므로 **모두** 지웁니다.

> **중복을 지우는 이유:** 같은 문서가 Train과 Test에 함께 들어가면 모델이 "외운 답"을 맞히는 셈이라 점수가 부풀려집니다(데이터 누수, 17장). 20 Newsgroups에는 같은 글을 여러 게시판에 올린 크로스포스팅이 실제로 있습니다.

In [14]:
step2 = step2.assign(norm=step2["text"].str.split().str.join(" "))   # 연속 공백을 하나로

n_labels = step2.groupby("norm")["label"].transform("nunique")       # 같은 본문의 레이블 종류 수
conflict = n_labels > 1
print("레이블이 엇갈린 문서:\n", step2.loc[conflict, ["text", "label"]], sep="")
step3 = step2[~conflict]
log.append(("레이블 충돌 중복 제거", len(step3)))

dup = step3["norm"].duplicated(keep="first")                         # 두 번째부터 True
print("\n중복으로 지울 문서:\n", step3.loc[dup, ["text", "label"]], sep="")
step4 = step3[~dup]
log.append(("중복 제거 (공백 차이 무시)", len(step4)))

레이블이 엇갈린 문서:
              text label
6  삼성전자 신형 스마트폰 공개  IT과학
7  삼성전자 신형 스마트폰 공개    경제

중복으로 지울 문서:
              text label
4  손흥민, 시즌 10호골 폭발   스포츠
5  누리호  4차 발사   성공  IT과학


`transform("nunique")`는 그룹별 값(레이블 종류 수)을 **원래 행마다** 다시 붙여 줍니다. 그래서 결과를 바로 행 필터 조건으로 쓸 수 있습니다.

### ④ 인덱스 다시 매기기
행을 지우면 인덱스에 구멍이 생깁니다(0, 1, 2, 3, 6, ...). 인덱스를 문서 ID로 쓰려면 `reset_index(drop=True)`로 0부터 다시 매깁니다.

In [15]:
clean = step4.drop(columns="norm").reset_index(drop=True)
print("정리 전 인덱스:", list(step4.index))
print("정리 후 인덱스:", list(clean.index))

print()
for name, n in log:
    print(f"{name:<24} {n:>3}건")
clean

정리 전 인덱스: [0, 1, 2, 3, 11, 12, 13]
정리 후 인덱스: [0, 1, 2, 3, 4, 5, 6]

원본                        15건
본문 또는 레이블 결측 제거           12건
빈 문서 제거                   11건
레이블 충돌 중복 제거               9건
중복 제거 (공백 차이 무시)           7건


,text,label
0,"손흥민, 시즌 10호골 폭발",스포츠
1,"코스피 2,600선 회복…외국인 순매수",경제
2,"정부, 반도체 지원 예산 확대",경제
3,누리호 4차 발사 성공,IT과학
4,"국회, 예산안 처리 합의",정치
5,프로야구 개막전 매진 행렬,스포츠
6,폭염 특보 전국 확대,사회


### 실제 데이터에도 적용
같은 과정을 20 Newsgroups에 적용해 봅니다. `remove=...`로 헤더와 인용문을 지우면 **본문이 아예 없어지는 글**이 생깁니다.

In [16]:
def clean_corpus(df: pd.DataFrame) -> tuple[pd.DataFrame, list[tuple[str, int]]]:
    log = [("원본", len(df))]
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.strip() != ""]
    log.append(("결측·빈 문서 제거", len(df)))
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    log.append(("레이블 충돌 중복 제거", len(df)))
    df = df[~norm.duplicated(keep="first")]
    log.append(("중복 제거", len(df)))
    return df.reset_index(drop=True), log


news_clean, news_log = clean_corpus(news[["text", "label"]])
for name, n in news_log:
    print(f"{name:<20} {n:>5,}건")
print("\n카테고리별:\n", news_clean["label"].value_counts().sort_index(), sep="")

원본                   4,941건
결측·빈 문서 제거           4,825건
레이블 충돌 중복 제거         4,821건
중복 제거                4,798건

카테고리별:
label
comp.graphics         954
comp.windows.x        961
rec.sport.baseball    956
rec.sport.hockey      974
sci.space             953
Name: count, dtype: int64


빈 문서가 100건 넘게 빠지고, 중복도 일부 빠집니다. 정리 후에도 문서가 수천 건이고 카테고리가 고르므로 과제 조건(500건 이상, 2개 이상 카테고리)을 충분히 만족합니다.

## 5.6 결과 저장

In [17]:
out_path = WORK / "news_clean.csv"
news_clean.to_csv(out_path, index=False, encoding="utf-8")   # index=False: 인덱스를 열로 저장하지 않음
reloaded = pd.read_csv(out_path)
print(out_path.name, reloaded.shape)
print("저장 전과 같은가:", reloaded.equals(news_clean))

news_clean.csv

 (4798, 2)
저장 전과 같은가: True


> 대용량 원본을 매번 정리하기 번거로우면 정리한 결과를 `data/` 아래에 저장해 두고 재사용합니다. 이 폴더는 `.gitignore`에 넣어 git에는 올리지 않습니다(1장).

## ✍️ 실습: 정리 전후 분포 표

`messy`(5.2절의 지저분한 데이터)와 `clean`(5.5절 결과)을 이용해 아래와 같은 표를 만드세요.

| label | 정리 전 | 정리 후 | 제거 |
|---|---|---|---|

- 정리 전 레이블이 결측인 행은 `"(결측)"`이라는 레이블로 셉니다(`fillna`).
- 정리 후에 없는 레이블은 0으로 채웁니다.
- 정리 전 문서 수가 많은 순서로 정렬합니다.

In [18]:
# 여기에 작성하세요

**정답**

In [19]:
before = messy["label"].fillna("(결측)").value_counts()
after = clean["label"].value_counts()

table = pd.DataFrame({"정리 전": before, "정리 후": after}).fillna(0).astype(int)
table["제거"] = table["정리 전"] - table["정리 후"]
table.sort_values("정리 전", ascending=False)

,정리 전,정리 후,제거
label,,,
IT과학,3,1,2
경제,3,2,1
사회,3,1,2
스포츠,3,2,1
정치,2,1,1
(결측),1,0,1


두 Series를 DataFrame으로 묶으면 **인덱스(레이블 이름)를 기준으로 자동 정렬**되어 같은 레이블끼리 맞춰집니다. 한쪽에만 있는 레이블은 `NaN`이 되므로 `fillna(0)`으로 채웁니다.

## 🧐 퀴즈

**Q1.** `df[df["label"] == "a" or df["label"] == "b"]`가 에러가 나는 이유와 올바른 코드는?
<details><summary>▶ 정답 보기</summary>

`or`는 값 하나의 참/거짓을 판단하는데, 비교 결과는 True/False가 여러 개인 Series라 판단할 수 없습니다. `df[(df["label"] == "a") | (df["label"] == "b")]` 또는 `df[df["label"].isin(["a", "b"])]`를 씁니다.
</details>

**Q2.** `dropna()`를 했는데도 빈 문서가 남을 수 있는 이유는?
<details><summary>▶ 정답 보기</summary>

`"   "`처럼 공백만 있는 문자열은 결측(NaN)이 아니기 때문입니다. `df["text"].str.strip() == ""`로 따로 찾아 지워야 합니다. 전처리(불용어 제거) 뒤에 토큰이 하나도 남지 않는 문서도 비슷한 문제를 일으킵니다.
</details>

**Q3.** 같은 본문이 Train과 Test에 함께 들어가면 어떤 문제가 생기나요?
<details><summary>▶ 정답 보기</summary>

모델이 Train에서 본 문서를 Test에서 다시 만나므로 실제 일반화 성능보다 점수가 높게 나옵니다(데이터 누수). 그래서 분할 전에 중복을 제거합니다.
</details>

**Q4.** `groupby("label").sample(n=1000, random_state=42)`에서 `random_state`를 고정하는 이유는?
<details><summary>▶ 정답 보기</summary>

다시 실행해도 같은 문서가 뽑혀야 결과(정확도, 오분류 사례 등)를 재현할 수 있기 때문입니다. 과제도 Train/Test 분할에서 `random_state` 고정을 요구합니다.
</details>

**Q5.** 행을 지운 뒤 `reset_index(drop=True)`를 하는 이유는?
<details><summary>▶ 정답 보기</summary>

지운 행 때문에 인덱스에 빈 번호가 생깁니다. 인덱스를 문서 ID로 쓰고, NumPy 행렬의 행 번호와 맞추려면 0부터 빈틈없이 다시 매겨야 합니다. `drop=True`는 옛 인덱스를 열로 남기지 않게 합니다.
</details>

## ✅ 체크포인트
- [ ] TSV, JSON, sklearn 데이터셋을 DataFrame으로 불러올 수 있다.
- [ ] 조건 필터에서 `&`, `|`, `isin`을 쓸 수 있다.
- [ ] `value_counts`로 카테고리 분포를 확인할 수 있다.
- [ ] `groupby(...).sample(...)`로 카테고리별 같은 수를 재현 가능하게 뽑을 수 있다.
- [ ] 결측, 빈 문서, 공백 차이 중복, 레이블 충돌 중복을 지우고 단계별 건수를 기록할 수 있다.

## 🔗 과제 연결과 다음 장
- 이 장의 `clean_corpus` 흐름은 7장에서 만드는 데이터 로딩 함수(`data.py`)의 뼈대입니다.
- 단계별 건수 기록과 분포 표는 리포트의 데이터 설명 부분에 씁니다.
- **6장. Matplotlib/Seaborn 최소한**에서 카테고리 분포를 막대그래프로, 혼동 행렬을 히트맵으로 그립니다.